## Encode it yourself: a mini `01_encodings` run

`01_inside_the_encoder.ipynb` opened up what a single forward pass returns and how pooling turns it into one vector. This notebook has you rebuild that pipeline yourself, end to end, at a tiny scale: pick a handful of PathMNIST patches, embed them with both encoders using the same `embed_images` helper the real pipeline uses, save the result to disk in the same format, and sanity-check it.

This is deliberately small -- 20 patches, not 1000+ -- so you can run every cell and inspect every intermediate result yourself before trusting `00_preparation/01_encodings/encode_pathmnist.ipynb` to do the same thing unattended at full scale.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import os

os.environ["HF_HOME"] = "/home/shared/.cache/huggingface"

In [ ]:
import sys

sys.path.append("/home/shared/helper/")

import torch

from vfm_encoders import load_uni2, load_virchow2  # noqa: E402

### 0. Setup

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. Load the encoders

Same as `01_inside_the_encoder.ipynb` -- `embed_dim` differs (1536 vs. 2560) because of the pooling choice you inspected there, not model size alone.

In [ ]:
encoders = {
    "uni2-h": load_uni2(device),
    "virchow2": load_virchow2(device),
}
for enc in encoders.values():
    n_params = sum(p.numel() for p in enc.model.parameters())
    print(f"{enc.name:10s} embed_dim={enc.embed_dim:5d}  params={n_params / 1e6:.0f}M")

### Exercise: pick a handful of patches

Load the same PathMNIST subset as before (`/home/shared/data/pathmnist/pathmnist_224_subset1000.npz`, `train_images`/`train_labels`). Take **20** patches -- any slice or sample is fine, but try to get at least 2-3 different tissue classes in the mix rather than 20 consecutive (likely same-class) patches, since the sanity check later wants some class variety. Convert them to a list of RGB `PIL.Image`s (`embed_images` expects that, not raw arrays) and keep their labels alongside.

### Exercise: encode your patches

Use `embed_images(encoder, images, device=device)` from `vfm_encoders` to embed your 20 images with **both** encoders (loop over the `encoders` dict from above). Store each model's `(20, embed_dim)` array. Before you run it: given what you saw in the previous notebook, what shape do you expect back for each encoder?

### Exercise: save your mini encodings

Save both encodings plus your labels into a single `.npz`, e.g. `~/data/pathmnist/encodings/mini_handson.npz` (`Path.home() / "data" / "pathmnist" / "encodings" / "mini_handson.npz"`) -- one array per model plus the labels, the same shape of layout `00_preparation/01_encodings/encode_pathmnist.ipynb`'s own "Save encodings" cell produces (worth a peek there if you want the exact key-naming pattern, after you've had a go yourself).

### Exercise: sanity check

Reload the `.npz` you just wrote and check:

1. Shapes round-trip correctly (same `(20, embed_dim)` you saved).
2. Pick a pair of patches that share a label and a pair that don't. For each encoder, is cosine similarity higher for the same-label pair than the mismatched one? Try 2-3 pairs, not just one -- a single pair can go either way by chance at this tiny sample size.

This is a hand-rolled, N=20 preview of what `03_encoding_analysis/01_tissue_type_probing_pathmnist.ipynb` checks properly with a real train/test split and a fitted classifier -- don't read too much into it if it's noisy.

### Questions to think about

No code needed for these -- just worth having an answer before moving on:

- Virchow2's embedding is 2560-d even though its hidden size is 1280-d. What did concatenating CLS with mean-pooled patches buy you that CLS alone wouldn't? (Hint: think about what information each half captures -- overall gist vs. an average over spatial detail.)
- Both encoders drop register tokens before pooling or exposing patch tokens. If registers aren't CLS and aren't spatial patches, why would a ViT need them at all, and why exclude them here rather than keep them in the mean?
- If you reran your sanity check with N=200 instead of N=20, would you expect the same-label-more-similar pattern to hold up *better* or *worse*? What does that tell you about how much to trust conclusions drawn from a handful of patches versus a full probe?
- Every embedding in this notebook came from a **frozen** encoder -- nothing was fine-tuned on PathMNIST or any other dataset here. What does that imply about what today's numbers can and can't tell you about performance on a specific downstream task, like ISUP grading?

### Takeaways

- The full pipeline in `00_preparation/01_encodings/encode_pathmnist.ipynb` (and its CAMELYON17/PANDA counterparts) is exactly this notebook, scaled up: same `embed_images` call, same `.npz` save pattern, just chunked over many more patches and both models saved as separate files instead of one.
- `03_encoding_analysis/` is where the sanity check above gets done properly -- real train/test splits, a fitted probe, and metrics rather than eyeballing 2-3 cosine similarities.
- If anything in this notebook surprised you (a shape, a similarity score, an error), that's worth a second look before treating the large-scale runs' outputs as ground truth.